In [ ]:
!pip install google-play-scraper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from google_play_scraper import reviews, Sort

### Tentukan ID Aplikasi

In [ ]:
APP_ID       = "us.zoom.videomeetings"
APP_NAME     = "Zoom Workplace"
LANG         = "id"
COUNTRY      = "id"
TOTAL_TARGET = 10000
BATCH_SIZE   = 200
OUTPUT_RAW   = "zoom_raw.csv"

print(f"Konfigurasi Aplikasi")
print(f"  App ID  : {APP_ID}")
print(f"  Bahasa  : {LANG}  |  Negara : {COUNTRY}")
print(f"  Target  : {TOTAL_TARGET:,} ulasan\n")

Konfigurasi Aplikasi
  App ID  : us.zoom.videomeetings
  Bahasa  : id  |  Negara : id
  Target  : 10,000 ulasan



### Scraping Ulasan

In [ ]:
all_reviews = []
continuation_token = None

print("[TAHAP 3] Mulai scraping ulasan...\n")

while len(all_reviews) < TOTAL_TARGET:
    need = min(BATCH_SIZE, TOTAL_TARGET - len(all_reviews))

    result, continuation_token = reviews(
        APP_ID,
        lang=LANG,
        country=COUNTRY,
        sort=Sort.NEWEST,         # ambil yang terbaru
        count=need,
        continuation_token=continuation_token
    )


    if not result:
        print("Tidak ada ulasan tambahan yang bisa diambil.")
        break

    all_reviews.extend(result)
    print(f"Berhasil ambil {len(result)} ulasan | Total sementara: {len(all_reviews)}")

    if continuation_token is None or continuation_token.token is None:
        print("Continuation token habis.")
        break

print(f"\nTotal ulasan terkumpul: {len(all_reviews)}")

[TAHAP 3] Mulai scraping ulasan...

Berhasil ambil 200 ulasan | Total sementara: 200
Berhasil ambil 200 ulasan | Total sementara: 400
Berhasil ambil 200 ulasan | Total sementara: 600
Berhasil ambil 200 ulasan | Total sementara: 800
Berhasil ambil 200 ulasan | Total sementara: 1000
Berhasil ambil 200 ulasan | Total sementara: 1200
Berhasil ambil 200 ulasan | Total sementara: 1400
Berhasil ambil 200 ulasan | Total sementara: 1600
Berhasil ambil 200 ulasan | Total sementara: 1800
Berhasil ambil 200 ulasan | Total sementara: 2000
Berhasil ambil 200 ulasan | Total sementara: 2200
Berhasil ambil 200 ulasan | Total sementara: 2400
Berhasil ambil 200 ulasan | Total sementara: 2600
Berhasil ambil 200 ulasan | Total sementara: 2800
Berhasil ambil 200 ulasan | Total sementara: 3000
Berhasil ambil 200 ulasan | Total sementara: 3200
Berhasil ambil 200 ulasan | Total sementara: 3400
Berhasil ambil 200 ulasan | Total sementara: 3600
Berhasil ambil 200 ulasan | Total sementara: 3800
Berhasil ambil 200

### Simpan Raw CSV

In [ ]:
df = pd.DataFrame(all_reviews)

# Pilih kolom yang paling berguna
kolom_penting = [
    "reviewId",
    "userName",
    "score",
    "at",
    "content",
    "thumbsUpCount",
]

kolom_tersedia = [k for k in kolom_penting if k in df.columns]
df = df[kolom_tersedia]

df.to_csv(OUTPUT_RAW, index=False, encoding="utf-8-sig")
print(f"[File raw berhasil disimpan: {OUTPUT_RAW}")

[File raw berhasil disimpan: zoom_raw.csv


## Preprocessing

In [ ]:
df

,reviewId,userName,score,at,content,thumbsUpCount
0,8ee488e3-f584-467d-bb8e-373262d9df31,Jiffri Putra,5,2026-04-08 14:31:20,Keren,0
1,f47fea52-947b-44fe-89df-4ced65480b6c,Elisabet Ingan,5,2026-04-08 13:02:33,good,0
2,46ad5975-c4fb-4b6d-91a0-3d1d787ae524,Ardianto Sattu Padang,1,2026-04-08 09:59:18,"Ini makin anehh dan ribett, gimana cara bagi l...",0
3,ae847c96-a1d9-4868-9009-f7b2e8a9b84e,Mahdi Sahputra15,5,2026-04-08 05:54:59,luar biasa,0
4,cc75336a-c697-45be-b63e-4ad0049ecc5a,brenda wadjo,5,2026-04-08 02:55:02,terbaik,0
...,...,...,...,...,...,...
9995,bf5bcc64-67e5-4133-b96f-4ea36c59e11b,Sestri Plg,5,2024-01-21 11:31:53,👍👍,0
9996,6db15ad6-8499-4c0d-a9f1-b023b2cb9d0f,Haikal Hidayatullah,5,2024-01-21 11:14:06,Aplikasi ini sangat membantu dalam belajar men...,0
9997,6447b921-c069-44e4-a150-d61f9e6ad67a,Rury Rhythm,1,2024-01-21 10:16:18,Apk cacat njir suaranya aja gk keluar waktu zo...,3
9998,56fef29d-1632-4056-b0de-f1de6320ee62,Yudi Pramana,5,2024-01-21 09:42:19,👍,0


In [ ]:
df = df.drop(columns=["thumbsUpCount"])

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   reviewId  10000 non-null  object        
 1   userName  10000 non-null  object        
 2   score     10000 non-null  int64         
 3   at        10000 non-null  datetime64[ns]
 4   content   9999 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 390.8+ KB


In [ ]:
df[df['content'].isna()]

,reviewId,userName,score,at,content
3872,322481b2-f111-4f28-b64f-ee61399af334,Niar Amir,5,2025-04-13 06:36:38,None


In [ ]:
df = df.dropna()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9999 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   reviewId  9999 non-null   object        
 1   userName  9999 non-null   object        
 2   score     9999 non-null   int64         
 3   at        9999 non-null   datetime64[ns]
 4   content   9999 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 468.7+ KB


In [ ]:
df

,reviewId,userName,score,at,content
0,8ee488e3-f584-467d-bb8e-373262d9df31,Jiffri Putra,5,2026-04-08 14:31:20,Keren
1,f47fea52-947b-44fe-89df-4ced65480b6c,Elisabet Ingan,5,2026-04-08 13:02:33,good
2,46ad5975-c4fb-4b6d-91a0-3d1d787ae524,Ardianto Sattu Padang,1,2026-04-08 09:59:18,"Ini makin anehh dan ribett, gimana cara bagi l..."
3,ae847c96-a1d9-4868-9009-f7b2e8a9b84e,Mahdi Sahputra15,5,2026-04-08 05:54:59,luar biasa
4,cc75336a-c697-45be-b63e-4ad0049ecc5a,brenda wadjo,5,2026-04-08 02:55:02,terbaik
...,...,...,...,...,...
9995,bf5bcc64-67e5-4133-b96f-4ea36c59e11b,Sestri Plg,5,2024-01-21 11:31:53,👍👍
9996,6db15ad6-8499-4c0d-a9f1-b023b2cb9d0f,Haikal Hidayatullah,5,2024-01-21 11:14:06,Aplikasi ini sangat membantu dalam belajar men...
9997,6447b921-c069-44e4-a150-d61f9e6ad67a,Rury Rhythm,1,2024-01-21 10:16:18,Apk cacat njir suaranya aja gk keluar waktu zo...
9998,56fef29d-1632-4056-b0de-f1de6320ee62,Yudi Pramana,5,2024-01-21 09:42:19,👍


In [ ]:
df['score'].unique()

array([5, 1, 3, 4, 2])

### Label Sentimen

In [ ]:
def label_sentimen(score):
    if score >= 4:
        return "positif"
    elif score == 3:
        return "netral"
    else:
        return "negatif"

df["sentimen"] = df["score"].apply(label_sentimen)

### TF-IDF

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('stopwords')
nltk.download('vader_lexicon')


# Preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['clean'] = df['content'].astype(str).apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
# Stopwords
stop_words = set(stopwords.words('indonesian'))

df['clean'] = df['clean'].apply(
    lambda x: " ".join([word for word in x.split() if word not in stop_words])
)

In [ ]:
fitur_list = [
    'audio', 'suara', 'video', 'kamera',
    'login', 'masuk',
    'meeting', 'room',
    'screen', 'share',
    'background',
    'koneksi', 'jaringan', 'internet',
    'chat'
]

In [ ]:
df_pos = df[df['score'] >= 4]
df_neg = df[df['score'] <= 2]

In [ ]:
fitur_score = {}

for fitur in fitur_list:
    pos = pos_fitur[fitur]
    neg = neg_fitur[fitur]

    score = (pos + 1) / (neg + 1)
    fitur_score[fitur] = score

# Urutkan
sorted_fitur = sorted(fitur_score.items(), key=lambda x: x[1], reverse=True)

print(sorted_fitur)

[('suara', 2.76), ('chat', 2.0), ('meeting', 1.5666666666666667), ('video', 1.3529411764705883), ('audio', 0.8333333333333334), ('koneksi', 0.6923076923076923), ('share', 0.5833333333333334), ('screen', 0.5555555555555556), ('room', 0.5), ('internet', 0.391304347826087), ('jaringan', 0.3108108108108108), ('background', 0.23446893787575152), ('masuk', 0.16964285714285715), ('kamera', 0.09090909090909091), ('login', 0.060810810810810814)]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000)
X = tfidf.fit_transform(df["clean"])
y = df["sentimen"]

## Prediksi Review Baru

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.845
              precision    recall  f1-score   support

     negatif       0.74      0.81      0.78       571
      netral       0.00      0.00      0.00       117
     positif       0.89      0.93      0.91      1312

    accuracy                           0.84      2000
   macro avg       0.54      0.58      0.56      2000
weighted avg       0.80      0.84      0.82      2000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
contoh = ["ngelag"]

contoh_clean = [clean_text(x) for x in contoh]
contoh_vec = tfidf.transform(contoh_clean)

prediksi = model.predict(contoh_vec)
print(prediksi)

['negatif']
